In [4]:
!pip install gradio plotly scikit-learn

In [8]:
# ================== IMPORTS ==================
import pandas as pd
import numpy as np
import gradio as gr
import plotly.express as px
import re
import random

# ================== LOAD DATA ==================
raw = pd.read_excel('Mercatus dataset.xlsx', header=None)

df = pd.DataFrame({
    'date': raw.iloc[:,0],
    'crop': raw.iloc[:,1],
    'market': raw.iloc[:,2],
    'price': raw.iloc[:,5]
})

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['crop'] = df['crop'].astype(str).str.lower().str.strip()
df['price'] = pd.to_numeric(df['price'], errors='coerce')

df = df.dropna()
df = df[df['crop'] != 'crop']
df = df[df['crop'] != 'redgram']

crops = sorted(df['crop'].unique())

# ================== FORECAST ==================
def forecast_crop(crop):

    data = df[df['crop'] == crop]
    data = data.groupby('date')['price'].mean().sort_index()

    base = data.values

    trend = np.linspace(base[-1], base[-1]*1.2, 24)
    seasonal = 0.1 * base.mean() * np.sin(np.linspace(0, 4*np.pi, 24))

    pred = trend + seasonal

    months = pd.date_range(start="2026-04-01", periods=24, freq='MS')

    best_idx = np.argmax(pred)
    worst_idx = np.argmin(pred)

    table = pd.DataFrame({
        "Month": months.strftime("%B %Y"),
        "Price (₹/qtl)": np.round(pred,2)
    })

    fig = px.line(
        x=months,
        y=pred,
        markers=True,
        title=f"{crop.upper()} Price Forecast",
        labels={"x":"Month","y":"Price per Quintal (₹)"}
    )

    text = f"""
🌾 Crop: {crop.upper()}

💰 Best Selling Month: {months[best_idx].strftime('%B %Y')} → ₹{round(pred[best_idx],2)}

⚠️ Avoid Glut Season: {months[worst_idx].strftime('%B %Y')} → ₹{round(pred[worst_idx],2)}
"""

    return text, fig, table


# ================== MARKET ==================
def market_suggestion(crop):

    data = df[df['crop'] == crop]
    markets = data.groupby('market')['price'].mean().sort_values(ascending=False)

    top = markets.head(3)

    return "\n".join([f"🏪 {m} → ₹{round(p,2)}" for m,p in top.items()])


# ================== BANKER CROPS ==================
def banker_crops():

    stability = {}

    for crop in crops:
        data = df[df['crop']==crop]
        series = data.groupby('date')['price'].mean()
        stability[crop] = series.std()

    stable = sorted(stability, key=stability.get)[:5]

    fig = px.bar(
        x=stable,
        y=[stability[c] for c in stable],
        labels={
            "x": "Crops",
            "y": "Price Fluctuation (Std Deviation)"
        },
        title="💰 Banker Crops (Least Fluctuating Crops Apr'26–Apr'28)"
    )

    return fig


# ================== CHATBOT (ULTIMATE) ==================
def chatbot(user_input, history):

    q = user_input.lower()
    words = re.findall(r'\b[a-z0-9]+\b', q)

    fallback = [
        "😅 Didn’t catch that… try 'tomato price june 2027'",
        "🤔 Ask like: best crop Dec 2026 or when to sell ragi",
        "🌾 I help with crops, prices, markets — try again!",
        "🚀 Try a smarter question to get richer 😎",
        "💰 Ask about crop + month + year",
        "🔄 Let’s try again!"
    ]

    try:

        # DETECT CROP
        detected_crop = None
        for crop in crops:
            if crop in words:
                detected_crop = crop
                break

        # DETECT MONTH
        months_map = {
            "jan":1,"january":1,
            "feb":2,"february":2,
            "mar":3,"march":3,
            "apr":4,"april":4,
            "may":5,
            "jun":6,"june":6,
            "jul":7,"july":7,
            "aug":8,"august":8,
            "sep":9,"september":9,
            "oct":10,"october":10,
            "nov":11,"november":11,
            "dec":12,"december":12
        }

        detected_month = None
        for m in months_map:
            if m in q:
                detected_month = months_map[m]
                break

        # DETECT YEAR
        detected_year = None
        for w in words:
            if w.isdigit() and len(w) == 4:
                detected_year = int(w)

        # PRICE QUERY
        if any(x in words for x in ["price","rate","value","cost"]):

            if detected_crop:

                _, _, table = forecast_crop(detected_crop)

                if detected_month and detected_year:

                    start_year = 2026
                    start_month = 4

                    idx = (detected_year - start_year)*12 + (detected_month - start_month)

                    if 0 <= idx < 24:
                        price = table.iloc[idx]["Price (₹/qtl)"]
                        month_name = table.iloc[idx]["Month"]

                        response = f"📊 {detected_crop.upper()} price in {month_name} is ₹{price}/qtl."
                    else:
                        response = "⚠️ Prediction available only from Apr 2026 to Apr 2028."

                else:
                    peak_idx = table["Price (₹/qtl)"].idxmax()
                    response = f"📈 {detected_crop.upper()} peaks in {table.iloc[peak_idx]['Month']}."

            else:
                response = "Tell me the crop name."

        # SELL
        elif any(x in words for x in ["sell","selling","harvest","when"]):

            if detected_crop:
                _, _, table = forecast_crop(detected_crop)
                idx = table["Price (₹/qtl)"].idxmax()

                response = f"📅 Sell {detected_crop.upper()} in {table.iloc[idx]['Month']} for max profit."
            else:
                response = "Which crop?"

        # BEST CROP
        elif any(x in words for x in ["best","profit","profitable","grow"]):

            best_crop = None
            best_price = -1

            for crop in crops:
                _, _, table = forecast_crop(crop)
                max_price = table["Price (₹/qtl)"].max()

                if max_price > best_price:
                    best_price = max_price
                    best_crop = crop

            response = f"🔥 {best_crop.upper()} is the most profitable crop."

        # MARKET
        elif "market" in words:

            if detected_crop:
                markets = df[df['crop']==detected_crop].groupby('market')['price'].mean().sort_values(ascending=False)
                top = markets.head(3)

                response = "🏪 Best markets:\n" + "\n".join(
                    [f"{m} → ₹{round(p,2)}" for m,p in top.items()]
                )
            else:
                response = "Tell me the crop."

        # GENERAL
        elif detected_crop:

            _, _, table = forecast_crop(detected_crop)
            idx = table["Price (₹/qtl)"].idxmax()

            response = f"📊 {detected_crop.upper()} performs best in {table.iloc[idx]['Month']}."

        else:
            response = random.choice(fallback)

    except:
        response = random.choice(fallback)

    history.append((user_input, response))
    return history, history


# ================== UI ==================
with gr.Blocks(theme=gr.themes.Soft()) as app:

    with gr.Tab("📄 About Mercatus"):
        gr.Markdown("""
# 🌾 Mercatus: Market Intelligence for Smart Farmers

### 💡 Banker Crop Ideology:
Top 5 crops whose prices fluctuate the least across Apr 2026–Apr 2028.

### 👨‍🌾 Group 19:
**Heramba V S (AMB2089) 🤖**
Hari Pavan Yadav (AMB2080)
Rahul Metre (AMB2198)
""")

    with gr.Tabs():

        with gr.Tab("📊 Dashboard"):
            crop = gr.Dropdown(crops)

            text = gr.Textbox()
            graph = gr.Plot()
            table = gr.Dataframe()

            crop.change(forecast_crop, crop, [text, graph, table])

            btn = gr.Button("Show Markets")
            market = gr.Textbox()

            btn.click(market_suggestion, crop, market)

        with gr.Tab("📈 Banker Crops"):
            plot = gr.Plot()
            gr.Button("Analyze").click(banker_crops, outputs=plot)

        with gr.Tab("🤖 AI Advisor"):
            chatbot_ui = gr.Chatbot()
            msg = gr.Textbox()

            def respond(message, chat_history):
                chat_history, _ = chatbot(message, chat_history)
                return "", chat_history

            msg.submit(respond, [msg, chatbot_ui], [msg, chatbot_ui])

app.launch()

/tmp/ipykernel_12663/3681140249.py:19: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

/tmp/ipykernel_12663/3681140249.py:247: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.

/tmp/ipykernel_12663/3681140249.py:283: UserWarning:

You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.

/tmp/ipykernel_12663/3681140249.py:283: DeprecationWarning:

The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disabl

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9c5df502198804f3c3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
